# 01 — Test risk-engine (état container + healthcheck + env vars)

Smoke test du container `fxvol-risk` (service `risk-engine` côté compose) — étape 1/5. Valide les **fondations** avant de tester les outputs Redis et le pipeline WS dans les notebooks suivants : container UP, healthcheck heartbeat-based passe healthy, env vars correctement injectées.

**Couvre** :
1. Container présent et `running`
2. Docker healthcheck = `healthy` (= `heartbeat:risk_engine` < 30s en Redis, cf. `docker-compose.yml § risk-engine`)
3. Image attendue (`fx-options-risk-engine:local`) + `StartedAt` + restart count raisonnable
4. Env vars critiques injectées : `SERVICE_NAME=risk_engine`, `IB_HOST=ib-gateway`, `IB_PORT=4002`, `IB_CLIENT_ID=3`, `REDIS_URL=redis://redis:6379/0`

**Préreq** :
- Stack démarrée avec profile `engines` + `ib` : `docker compose --profile engines --profile ib up -d`
- `redis` healthy (le healthcheck de risk-engine dépend de Redis pour vérifier le heartbeat)
- `vol-engine` started (dependency listée dans compose, mais risk-engine peut tourner sans surface — le cycle skip silencieusement, le heartbeat est quand même poussé)
- `market-data` healthy (idem, dependency indirecte via le besoin de `latest_spot:EURUSD` pour les cycles ; sans, cycles skip mais heartbeat OK)
- `ib-gateway` healthy (la connexion IB est ouverte mais pas utilisée pour des reqMktData ; si ib-gateway down, l'engine retry en backoff sans bloquer)
- ~30-60s d'attente après `up` pour que start_period (30s) soit fini et qu'au moins 1 heartbeat ait été poussé

**Référence** : `docker-compose.yml § risk-engine`, `src/services/risk/main.py`, `src/services/risk/engine.py` (cycle 2s).

## Setup

Note : le container Docker s'appelle `fxvol-risk` (sans `-engine`), même si le service compose est `risk-engine`. C'est un choix de naming utilisateur récent — alignement avec `fxvol-market-data` (sans `-engine` aussi).

In [61]:
import subprocess

CONTAINER = "fxvol-risk"
results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

def docker_inspect(fmt):
    out = subprocess.run(
        ["docker", "inspect", "-f", fmt, CONTAINER],
        capture_output=True, text=True,
    )
    return out.stdout.strip()

print(f"target container = {CONTAINER}")

target container = fxvol-risk


## 1. Container présent et `running`

**Ce que tu dois voir** : state = `running`. Si `exited` → crash au boot (logs avec `docker logs fxvol-risk --tail 100`). Si `restarting` → crashloop. Si `<not found>` → stack pas démarrée avec profile `engines`.

In [62]:
print("== 1. container state ==")

state = docker_inspect("{{.State.Status}}")
record("docker container state", state == "running", state or "<not found>")

restart_count = docker_inspect("{{.RestartCount}}")
record("restart count raisonnable (≤ 3)",
       restart_count.isdigit() and int(restart_count) <= 3,
       f"restart count = {restart_count}")

== 1. container state ==
  [OK  ] docker container state  | running
  [OK  ] restart count raisonnable (≤ 3)  | restart count = 0


## 2. Docker healthcheck = `healthy`

Le healthcheck (cf. fix R9 sur la compose, parsing ISO-8601) lit `heartbeat:risk_engine` dans Redis et vérifie que l'âge est < 30s. Donc `healthy` ⇒ engine pousse activement des heartbeats (cycle 2s).

**Note** : contrairement à market-data dont le healthcheck dépend des ticks IB, ici le heartbeat est poussé **inconditionnellement** à chaque cycle, **même si** spot/surface manquent en Redis (le cycle skip le calcul Greeks mais incrémente quand même le heartbeat). Donc `healthy` = engine en boucle, pas forcément que les Greeks sont publiés. Les Greeks sont validés au notebook 02.

In [63]:
print("== 2. healthcheck ==")

health = docker_inspect("{{.State.Health.Status}}")
record("docker healthcheck", health == "healthy", health or "<no healthcheck>")

last_log = docker_inspect("{{(index .State.Health.Log 0).Output}}")
if last_log:
    print(f"  [INFO] dernière sortie healthcheck : {last_log[:200]!r}")

== 2. healthcheck ==
  [FAIL] docker healthcheck  | unhealthy


## 3. Image + StartedAt (info)

Pas un test pass/fail — juste de l'info pour le diag.

In [64]:
print("== 3. image + uptime info ==")

image = docker_inspect("{{.Config.Image}}")
started_at = docker_inspect("{{.State.StartedAt}}")
print(f"  [INFO] image      : {image}")
print(f"  [INFO] StartedAt  : {started_at}")

expected_prefix = "fx-options-risk-engine"
record(f"image préfixe '{expected_prefix}'",
       expected_prefix in image,
       image)

== 3. image + uptime info ==
  [INFO] image      : fx-options-risk-engine:local
  [INFO] StartedAt  : 2026-04-28T15:00:32.324081111Z
  [OK  ] image préfixe 'fx-options-risk-engine'  | fx-options-risk-engine:local


## 4. Env vars critiques injectées

Filtre positif explicite (cf. mémoire `feedback_docker_inspect_env_leak`). Les 5 vars listées sont toutes non-secrètes, donc afficher leurs valeurs ne pose pas de risque.

**ClientId 3 critique** : doit être différent de market-data (1) et vol-engine (2) pour que les 3 engines puissent coexister sur Gateway. Si jamais quelqu'un override sur 1 par accident, les engines se kicker mutuellement avec erreur 326.

In [65]:
print("== 4. env vars critiques (filtrées non-secret) ==")

expected = {
    "SERVICE_NAME": "risk_engine",
    "IB_HOST": "ib-gateway",
    "IB_PORT": "4002",
    "IB_CLIENT_ID": "3",
    "REDIS_URL": "redis://redis:6379/0",
}

key_pattern = "^(" + "|".join(expected) + ")="
out = subprocess.run(
    ["docker", "exec", CONTAINER, "sh", "-c",
     f'env | grep -E "{key_pattern}"'],
    capture_output=True, text=True,
)
lines = out.stdout.strip().splitlines()
actual = dict(line.split("=", 1) for line in lines if "=" in line)

for key, expected_val in expected.items():
    actual_val = actual.get(key, "<missing>")
    record(f"{key} = {expected_val!r}",
           actual_val == expected_val,
           actual_val if actual_val != expected_val else "")

== 4. env vars critiques (filtrées non-secret) ==
  [OK  ] SERVICE_NAME = 'risk_engine'
  [OK  ] IB_HOST = 'ib-gateway'
  [OK  ] IB_PORT = '4002'
  [OK  ] IB_CLIENT_ID = '3'
  [OK  ] REDIS_URL = 'redis://redis:6379/0'


## Récap final

In [66]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 100)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 100)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK container risk-engine sain. Pass au notebook 02 (Redis outputs avec seed surface).")


LABEL                                                   STATUS  DETAIL
----------------------------------------------------------------------------------------------------
docker container state                                  OK      running
restart count raisonnable (≤ 3)                         OK      restart count = 0
docker healthcheck                                      FAIL    unhealthy
image préfixe 'fx-options-risk-engine'                  OK      fx-options-risk-engine:local
SERVICE_NAME = 'risk_engine'                            OK      
IB_HOST = 'ib-gateway'                                  OK      
IB_PORT = '4002'                                        OK      
IB_CLIENT_ID = '3'                                      OK      
REDIS_URL = 'redis://redis:6379/0'                      OK      
----------------------------------------------------------------------------------------------------

8 OK / 1 FAIL  (9 total)


## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| `state` = `<not found>` | Stack lancée sans profile `engines` | `docker compose --profile engines --profile ib up -d` |
| `state` = `restarting` ou restart count > 3 | Crashloop : engine plante au boot | `docker logs fxvol-risk --tail 100` ; suspects : `IB_HOST` mauvais, Redis URL incorrecte, ib-gateway pas encore healthy au boot |
| `healthcheck` = `starting` depuis > 60s | start_period dépassé, heartbeat pas pushé | engine pas en boucle ; vérifier logs |
| `healthcheck` = `unhealthy` | Heartbeat absent ou stale > 30s | `docker compose exec redis redis-cli GET heartbeat:risk_engine` ; si vide → engine arrêté en cycle |
| `IB_CLIENT_ID != "3"` | Override compose appliqué | OK si voulu (cohabitation custom), sinon revoir compose ; risque de collision avec market-data (1) ou vol-engine (2) |
| `image` ≠ `fx-options-risk-engine:*` | Image custom (release pinnée, registry distant) | Pas un fail si voulu, sinon revoir build |